Import statements

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

Creating a Spark Session

In [ ]:
spark = SparkSession.builder.appName("Employee Data Cleaning").getOrCreate()

In [ ]:
spark

Reading the csv into spark dataframe

In [ ]:
df=spark.read.csv('/content/Messy_Employee_dataset.csv',inferSchema=True,header=True)
df.show()

+-----------+----------+---------+----+-------------------+--------+----------+---------+--------------------+-----------+-----------------+-----------+
|Employee_ID|First_Name|Last_Name| Age|  Department_Region|  Status| Join_Date|   Salary|               Email|      Phone|Performance_Score|Remote_Work|
+-----------+----------+---------+----+-------------------+--------+----------+---------+--------------------+-----------+-----------------+-----------+
|    EMP1000|       Bob|    Davis|  25|  DevOps-California|  Active|  4/2/2021| 59767.65|bob.davis@example...|-1651623197|          Average|       true|
|    EMP1001|       Bob|    Brown|NULL|      Finance-Texas|  Active| 7/10/2020| 65304.66|bob.brown@example...|-1898471390|        Excellent|       true|
|    EMP1002|     Alice|    Jones|NULL|       Admin-Nevada| Pending| 12/7/2023|  88145.9|alice.jones@examp...|-5596363211|             Good|       true|
|    EMP1003|       Eva|    Davis|  25|       Admin-Nevada|Inactive|11/27/2021| 69

Validating the dataset

In [ ]:
df.printSchema()

root
 |-- Employee_ID: string (nullable = true)
 |-- First_Name: string (nullable = true)
 |-- Last_Name: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Department_Region: string (nullable = true)
 |-- Status: string (nullable = true)
 |-- Join_Date: string (nullable = true)
 |-- Salary: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- Phone: long (nullable = true)
 |-- Performance_Score: string (nullable = true)
 |-- Remote_Work: boolean (nullable = true)



In [ ]:
#replacing invalid values before casting
df = df.withColumn("Salary", when(col("Salary") == "N/A", None).otherwise(col("Salary")))


In [ ]:
#casting the salary to double
df = df.withColumn("Salary", col("Salary").cast("double"))
df.printSchema()

root
 |-- Employee_ID: string (nullable = true)
 |-- First_Name: string (nullable = true)
 |-- Last_Name: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Department_Region: string (nullable = true)
 |-- Status: string (nullable = true)
 |-- Join_Date: string (nullable = true)
 |-- Salary: double (nullable = true)
 |-- Email: string (nullable = true)
 |-- Phone: long (nullable = true)
 |-- Performance_Score: string (nullable = true)
 |-- Remote_Work: boolean (nullable = true)



In [ ]:
#Checking the number of rows and cols
print("Rows:",df.count())
print("Columns:",len(df.columns))
print(df.columns)


Rows: 1020
Columns: 12
['Employee_ID', 'First_Name', 'Last_Name', 'Age', 'Department_Region', 'Status', 'Join_Date', 'Salary', 'Email', 'Phone', 'Performance_Score', 'Remote_Work']


Cleaning the data

In [ ]:
#checking for null values
df.select([
    sum(
        when(col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in df.columns
]).show()

+-----------+----------+---------+---+-----------------+------+---------+------+-----+-----+-----------------+-----------+
|Employee_ID|First_Name|Last_Name|Age|Department_Region|Status|Join_Date|Salary|Email|Phone|Performance_Score|Remote_Work|
+-----------+----------+---------+---+-----------------+------+---------+------+-----+-----+-----------------+-----------+
|          0|         0|        0|211|                0|     0|        0|    24|    0|    0|                0|          0|
+-----------+----------+---------+---+-----------------+------+---------+------+-----+-----+-----------------+-----------+



In [ ]:
#checking the rows that have null in Age column
df.filter(col("Age").isNull()).show()

+-----------+----------+---------+----+-------------------+--------+----------+---------+--------------------+-----------+-----------------+-----------+
|Employee_ID|First_Name|Last_Name| Age|  Department_Region|  Status| Join_Date|   Salary|               Email|      Phone|Performance_Score|Remote_Work|
+-----------+----------+---------+----+-------------------+--------+----------+---------+--------------------+-----------+-----------------+-----------+
|    EMP1001|       Bob|    Brown|NULL|      Finance-Texas|  Active| 7/10/2020| 65304.66|bob.brown@example...|-1898471390|        Excellent|       true|
|    EMP1002|     Alice|    Jones|NULL|       Admin-Nevada| Pending| 12/7/2023|  88145.9|alice.jones@examp...|-5596363211|             Good|       true|
|    EMP1006|     Frank|    Jones|NULL|       Admin-Nevada|  Active|  4/3/2020| 96288.43|frank.jones@examp...|-4518376063|             Good|      false|
|    EMP1009|   Charlie|  Johnson|NULL|    DevOps-New York|  Active|  8/4/2022| 76

In [ ]:
#checking the rows that have null in Salary column
df.filter(col("Salary").isNull()).show()

+-----------+----------+---------+----+--------------------+--------+----------+------+--------------------+-----------+-----------------+-----------+
|Employee_ID|First_Name|Last_Name| Age|   Department_Region|  Status| Join_Date|Salary|               Email|      Phone|Performance_Score|Remote_Work|
+-----------+----------+---------+----+--------------------+--------+----------+------+--------------------+-----------+-----------------+-----------+
|    EMP1099|       Eva|    Jones|  25|    Admin-California|  Active|  1/9/2022|  NULL|eva.jones@example...|-2582386869|             Good|      false|
|    EMP1289|     Grace|   Garcia|  25|      Admin-New York|Inactive|10/24/2021|  NULL|grace.garcia@exam...|-2694500533|             Poor|      false|
|    EMP1329|     Heidi|   Garcia|  40|    Cloud Tech-Texas| Pending|  2/3/2023|  NULL|heidi.garcia@exam...|-8171752420|        Excellent|      false|
|    EMP1344|     Grace| Williams|  30|      DevOps-Florida|Inactive|  1/8/2022|  NULL|grace.w

In [ ]:
mean_age = df.select(avg("Age")).first()[0]
print(mean_age)

32.48454882571075


In [ ]:
mean_salary=df.select(avg("Salary")).first()[0]
print(mean_salary)

85155.05639558229


In [ ]:
#replacing the null values in Age & Salary columns with mean
df_filled=df.na.fill(int(mean_age),subset=["Age"])
df_filled=df_filled.na.fill(int(mean_salary),subset=["Salary"])
df_filled.show()

+-----------+----------+---------+---+-------------------+--------+----------+---------+--------------------+-----------+-----------------+-----------+
|Employee_ID|First_Name|Last_Name|Age|  Department_Region|  Status| Join_Date|   Salary|               Email|      Phone|Performance_Score|Remote_Work|
+-----------+----------+---------+---+-------------------+--------+----------+---------+--------------------+-----------+-----------------+-----------+
|    EMP1000|       Bob|    Davis| 25|  DevOps-California|  Active|  4/2/2021| 59767.65|bob.davis@example...|-1651623197|          Average|       true|
|    EMP1001|       Bob|    Brown| 32|      Finance-Texas|  Active| 7/10/2020| 65304.66|bob.brown@example...|-1898471390|        Excellent|       true|
|    EMP1002|     Alice|    Jones| 32|       Admin-Nevada| Pending| 12/7/2023|  88145.9|alice.jones@examp...|-5596363211|             Good|       true|
|    EMP1003|       Eva|    Davis| 25|       Admin-Nevada|Inactive|11/27/2021| 69450.99|

In [ ]:
#checking for duplicates
duplicates=df_filled.count()-df_filled.dropDuplicates().count()
duplicates

0

In [ ]:
#splitting the department_region into two separate columns and dropping the dept_reg
df_filled = df_filled.withColumn("Department", split(col("Department_Region"), "-")[0]) \
       .withColumn("Region", split(col("Department_Region"), "-")[1])
df_filled=df_filled.drop("Department_Region")

In [ ]:
df_filled.printSchema()

root
 |-- Employee_ID: string (nullable = true)
 |-- First_Name: string (nullable = true)
 |-- Last_Name: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Status: string (nullable = true)
 |-- Join_Date: string (nullable = true)
 |-- Salary: double (nullable = false)
 |-- Email: string (nullable = true)
 |-- Phone: long (nullable = true)
 |-- Performance_Score: string (nullable = true)
 |-- Remote_Work: boolean (nullable = true)
 |-- Department: string (nullable = true)
 |-- Region: string (nullable = true)



In [ ]:
df_filled.show()

+-----------+----------+---------+---+--------+----------+---------+--------------------+-----------+-----------------+-----------+----------+----------+
|Employee_ID|First_Name|Last_Name|Age|  Status| Join_Date|   Salary|               Email|      Phone|Performance_Score|Remote_Work|Department|    Region|
+-----------+----------+---------+---+--------+----------+---------+--------------------+-----------+-----------------+-----------+----------+----------+
|    EMP1000|       Bob|    Davis| 25|  Active|  4/2/2021| 59767.65|bob.davis@example...|-1651623197|          Average|       true|    DevOps|California|
|    EMP1001|       Bob|    Brown| 32|  Active| 7/10/2020| 65304.66|bob.brown@example...|-1898471390|        Excellent|       true|   Finance|     Texas|
|    EMP1002|     Alice|    Jones| 32| Pending| 12/7/2023|  88145.9|alice.jones@examp...|-5596363211|             Good|       true|     Admin|    Nevada|
|    EMP1003|       Eva|    Davis| 25|Inactive|11/27/2021| 69450.99|eva.davi

EDA

In [ ]:
#Employee distribution across the departments
df_filled.groupBy("Department").count().show()

+----------+-----+
|Department|count|
+----------+-----+
|     Sales|  178|
|        HR|  171|
|   Finance|  170|
|     Admin|  166|
|Cloud Tech|  146|
|    DevOps|  189|
+----------+-----+



In [ ]:
#Employee distribution across regions
df_filled.groupBy("Region").count().show()

+----------+-----+
|    Region|count|
+----------+-----+
|     Texas|  153|
|    Nevada|  169|
|  Illinois|  165|
|   Florida|  185|
|California|  187|
|  New York|  161|
+----------+-----+



In [ ]:
#Average salary across the departments
df_filled.groupBy("Department").agg(round(avg("Salary"),2).alias("Average Salary")).orderBy(desc("Average Salary")).show()

+----------+--------------+
|Department|Average Salary|
+----------+--------------+
|     Sales|      86867.33|
|    DevOps|      85993.25|
|        HR|      85466.19|
|     Admin|      85190.81|
|Cloud Tech|      84930.98|
|   Finance|       82274.9|
+----------+--------------+



In [ ]:
#Employee distribution across age
df_filled.groupBy("Age").count().orderBy(asc("Age")).show()

+---+-----+
|Age|count|
+---+-----+
| 25|  206|
| 30|  205|
| 32|  211|
| 35|  188|
| 40|  210|
+---+-----+



In [ ]:
#Employees & count of employees whose performance score is "Excellent"
df_filled.filter(col("Performance_Score")=="Excellent").show()
df_filled.filter(col("Performance_Score")=="Excellent").count()

+-----------+----------+---------+---+--------+----------+---------+--------------------+-----------+-----------------+-----------+----------+----------+
|Employee_ID|First_Name|Last_Name|Age|  Status| Join_Date|   Salary|               Email|      Phone|Performance_Score|Remote_Work|Department|    Region|
+-----------+----------+---------+---+--------+----------+---------+--------------------+-----------+-----------------+-----------+----------+----------+
|    EMP1001|       Bob|    Brown| 32|  Active| 7/10/2020| 65304.66|bob.brown@example...|-1898471390|        Excellent|       true|   Finance|     Texas|
|    EMP1008|     Frank|    Davis| 35|Inactive| 12/8/2023|115565.82|frank.davis@examp...|-4177656123|        Excellent|       true|     Admin|    Nevada|
|    EMP1009|   Charlie|  Johnson| 32|  Active|  8/4/2022| 76561.88|charlie.johnson@e...|-8156985699|        Excellent|       true|    DevOps|  New York|
|    EMP1012|     Heidi| Williams| 30|  Active|  3/5/2024| 89295.77|heidi.wi

267

In [ ]:
#ranking employees by salaries within each department
windowSpec=Window.partitionBy("Department").orderBy(desc("Salary"))
df_filled=df_filled.withColumn("Rank",rank().over(windowSpec))
df_filled.show()

+-----------+----------+---------+---+--------+----------+---------+--------------------+-----------+-----------------+-----------+----------+----------+----+
|Employee_ID|First_Name|Last_Name|Age|  Status| Join_Date|   Salary|               Email|      Phone|Performance_Score|Remote_Work|Department|    Region|Rank|
+-----------+----------+---------+---+--------+----------+---------+--------------------+-----------+-----------------+-----------+----------+----------+----+
|    EMP1103|     Alice|    Brown| 30|Inactive| 6/23/2020|119574.27|alice.brown@examp...|-3985547434|          Average|       true|     Admin|  Illinois|   1|
|    EMP1731|     Alice|    Smith| 35| Pending|  8/9/2024|119311.14|alice.smith@examp...|-2835129591|             Poor|       true|     Admin|California|   2|
|    EMP1068|       Bob|    Brown| 40| Pending|12/23/2022|119152.47|bob.brown@example...| -846684549|             Poor|       true|     Admin|   Florida|   3|
|    EMP1716|     Alice| Williams| 32|  Active

In [ ]:
df_filled.write.parquet("/content/data/cleaned_EmployeeData",mode="overwrite")

In [ ]:
cleaned_df = spark.read.parquet(
    "/content/data/cleaned_EmployeeData"
)
cleaned_df.show()

+-----------+----------+---------+---+--------+----------+---------+--------------------+-----------+-----------------+-----------+----------+----------+----+
|Employee_ID|First_Name|Last_Name|Age|  Status| Join_Date|   Salary|               Email|      Phone|Performance_Score|Remote_Work|Department|    Region|Rank|
+-----------+----------+---------+---+--------+----------+---------+--------------------+-----------+-----------------+-----------+----------+----------+----+
|    EMP1103|     Alice|    Brown| 30|Inactive| 6/23/2020|119574.27|alice.brown@examp...|-3985547434|          Average|       true|     Admin|  Illinois|   1|
|    EMP1731|     Alice|    Smith| 35| Pending|  8/9/2024|119311.14|alice.smith@examp...|-2835129591|             Poor|       true|     Admin|California|   2|
|    EMP1068|       Bob|    Brown| 40| Pending|12/23/2022|119152.47|bob.brown@example...| -846684549|             Poor|       true|     Admin|   Florida|   3|
|    EMP1716|     Alice| Williams| 32|  Active

In [ ]:
cleaned_df.printSchema()

root
 |-- Employee_ID: string (nullable = true)
 |-- First_Name: string (nullable = true)
 |-- Last_Name: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Status: string (nullable = true)
 |-- Join_Date: string (nullable = true)
 |-- Salary: double (nullable = true)
 |-- Email: string (nullable = true)
 |-- Phone: long (nullable = true)
 |-- Performance_Score: string (nullable = true)
 |-- Remote_Work: boolean (nullable = true)
 |-- Department: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Rank: integer (nullable = true)

